In [2]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "plate_well" # or None
random_seed = 42

condition_keys = "perturbation" # 数据集perturbation所对应的obs列名
dataset_name = "Sciplex3_test"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"
cov_config = { # adata中的covariant对应的obs列名
    "cell_line":{
        "type": "categorical",  #连续or离散 在adata的obs中添加{cov_name}_idx列，将categorical转化为数字，continuous做
        "control_ot": "groupwise",  #control是否要根据该covariant分层
        "perturbed_ot": "groupwise", #perturbed是否要根据该covariant分层
        "use_in_model": True, #是否作为模型输入
        "model_input_from": "control", #模型输入从哪来 
        "contain_in_condition_combined_keys":True, #是否要写入condition_combined_keys
        "condition_combined_from": "both", #condition_combined_keys从哪里提取（比如cell_type应该both中提取，dose_value应该从perturbed中提取） #最后会体现在如何写入
    },
    # "celltype":{
    #     "type": "categorical",
    #     "control_ot": "global",
    #     "perturbed_ot": "global",
    #     "use_in_model": True,
    #     "model_input_from": "control",
    #     "contain_in_condition_combined_keys":False,
    #     "condition_combined_from": None,
    # },
    "dose_value":{
        "type": "continuous",
        "control_ot": "global",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_input_from": "condition_combined_keys",
        "contain_in_condition_combined_keys":True,
        "condition_combined_from": "perturbed",
    },
    "time":{
        "type": "continuous",
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_input_from": "condition_combined_keys",
        "contain_in_condition_combined_keys":True,
        "condition_combined_from": "both",
    },
}

if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"

condition_rep_dict = pd.read_pickle("./data/processed/drug_embeddings_sciplex3_chembert.pkl")
condition_rep_dict = condition_rep_dict["drug_to_embedding"]

In [6]:
filePath = './data/raw/Sciplex3_test.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

AnnData object with n_obs × n_vars = 7627 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config'
    layers: 'counts'


In [7]:
adata.obs[control_key] = (adata.obs[condition_keys] == "control")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    7464
True      163
Name: count, dtype: int64


In [8]:
adata.obs[mass_deduct_keys] = adata.obs["plate"].astype(str) + "_" + adata.obs["well"].astype(str) #处理mass

## splitting

In [9]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.2
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    print(condition_list)
else:
    # 按condition分割 zeroshot
    n_test = max(1, int(len(condition_list) * test_ratio))
    test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()

['UNC0631', 'UNC1999', 'Lapatinib (GW-572016) Ditosylate', 'Entinostat (MS-275)', 'Gandotinib (LY2784544)', 'AICAR (Acadesine)', 'GSK1070916', 'AC480 (BMS-599626)', 'Thalidomide', 'PHA-680632', 'SRT2104 (GSK2245840)', 'ENMD-2076 L-(+)-Tartaric acid', 'Roxadustat (FG-4592)', 'SNS-314', 'CUDC-101', 'PD173074', 'WHI-P154', 'Decitabine', 'Tozasertib (VX-680, MK-0457)', 'Pirarubicin', 'ENMD-2076', 'FLLL32', 'Flavopiridol HCl', 'Obatoclax Mesylate (GX15-070)', 'Ivosidenib (AG-120)', 'Glesatinib?(MGCD265)', 'S-Ruxolitinib (INCB018424)', 'Mocetinostat (MGCD0103)', 'G007-LK', 'Lomustine', 'KW-2449', 'Tucidinostat (Chidamide)', 'Triamcinolone Acetonide', 'PJ34', 'AMG-900', 'Tubastatin A HCl', 'Anacardic Acid']
['IOX2', 'AZ 960', 'Resminostat', 'Fasudil (HA-1077) HCl', 'Ruxolitinib (INCB018424)', 'Tofacitinib (CP-690550) Citrate', 'INO-1001 (3-Aminobenzamide)', 'SRT3025 HCl', 'Busulfan', 'SL-327', 'Rigosertib (ON-01910)', 'Ramelteon', '2-Methoxyestradiol (2-MeOE2)', 'Rucaparib (AG-014699,PF-01367

In [10]:
del adata

## latent embedding

In [11]:
n_comps = 100
n_hidden = 1024
n_layers = 2
#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [12]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [13]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

[6.27858    5.0495524  1.9168108  1.5857215  1.645168   1.5704936
 1.1689501  1.213968   1.2274845  1.0144765  1.2569205  1.1603113
 1.0767416  1.1947519  0.99542236 1.1748184  1.0122947  1.0859402
 0.95045185 0.88577    1.0493556  0.97931415 1.0616724  0.9393487
 0.964272   0.8948828  0.924398   0.87520605 0.8216472  0.798256
 0.933789   0.9216317  0.9427908  0.95161676 0.9187407  0.940825
 0.9195647  0.899552   0.92196614 0.8629942  0.85492843 0.86749274
 0.8562299  0.8489572  0.84970677 0.7444489  0.8883408  0.9317501
 0.8164582  0.81356514 0.769427   0.9009341  0.8097197  0.7658775
 0.87689245 0.7678289  0.7840728  0.86959106 0.849925   0.84682953
 0.90700734 0.79991806 0.8575172  0.8573121  0.8358184  0.7429862
 0.8957763  0.73514    0.8898433  0.7088583  0.7289302  0.8865529
 0.67877257 0.60237837 0.6937889  0.7933766  0.7323243  0.6765009
 0.7022521  0.70317745 0.7845596  0.77099353 0.710149   0.70495975
 0.8095961  0.7535517  0.71555096 0.7688345  0.6736554  0.808274
 0.657132 

In [14]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [15]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/Sciplex3_test_42_0.2_True_X_pca_100_None
AnnData object with n_obs × n_vars = 163 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control', 'plate_well', 'cell_line_idx', 'dose_value_scaled', 'time_scaled', 'condition_combined'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config', 'pca', 'covariate_info'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 6000 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control', 'plate_w

In [16]:
adata_control.uns

{'hvg': {'flavor': 'seurat'},
 'log1p': {},
 'cov_config': {'cell_line': {'type': 'categorical',
   'control_ot': 'groupwise',
   'perturbed_ot': 'groupwise',
   'use_in_model': True,
   'model_input_from': 'control',
   'contain_in_condition_combined_keys': True,
   'condition_combined_from': 'both'},
  'dose_value': {'type': 'continuous',
   'control_ot': 'global',
   'perturbed_ot': 'groupwise',
   'use_in_model': True,
   'model_input_from': 'condition_combined_keys',
   'contain_in_condition_combined_keys': True,
   'condition_combined_from': 'perturbed'},
  'time': {'type': 'continuous',
   'control_ot': 'groupwise',
   'perturbed_ot': 'groupwise',
   'use_in_model': True,
   'model_input_from': 'condition_combined_keys',
   'contain_in_condition_combined_keys': True,
   'condition_combined_from': 'both'}},
 'pca': {'params': {'zero_center': False,
   'use_highly_variable': False,
   'mask_var': None,
   'layer': 'X_centered'},
  'variance': array([39.738014  , 25.82164   ,  4.18

In [53]:
adata_control.obs[condition_combined_keys].value_counts()

condition_combined
control_MCF7_control_24.0    7786
control_K562_control_24.0    3935
control_A549_control_24.0    3773
control_A549_control_72.0    2084
Name: count, dtype: int64

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()